# Lab — Binary Classification and Logistic Models

In this first lab, we explore binary. We gradually move from simple Gaussian mixture models to logistic regression, polynomial feature expansion, ROC analysis, and finally a home-made implementation of logistic regression.

**Objectives:**
- Understand the Bayes classifier in the Gaussian mixture model.
- Implement and visualize the Bayes decision boundary.
- Build a Linear Discriminant Analysis (LDA) classifier.
- Compare LDA to logistic regression.
- Enrich logistic regression with polynomial features.
- Evaluate models using ROC curves and AUC.
- Implement logistic regression *from scratch* with gradient descent.


## 1. Gaussian Mixture and Bayes Classifier

We begin with a simple synthetic binary classification problem:
a **mixture of two Gaussians** with equal covariance matrix.  
In this setting, the **Bayes classifier** is known in closed form and is itself a *linear* classifier.

### Task
1. Implement a class that computes the Bayes decision rule.
2. Display the decision boundary, and experiment with several values of `p` and `rho` (correlation).

The function below will perform the plotting for you.

Recall: given $p = \mathbb{P}(Y=1)$ is the class prior probability,
$\mu_0,\mu_1 \in \mathbb{R}^d$ are the class-conditional means,
and $\Sigma \in \mathbb{R}^{d\times d}$ is the common covariance matrix
(assumed symmetric positive definite),
\begin{aligned}
w &= \Sigma^{-1}(\mu_1-\mu_0),\\[0.6em]
b &= \log\frac{p}{1-p}
\;-\;\frac12\Big(\mu_1^\top\Sigma^{-1}\mu_1-\mu_0^\top\Sigma^{-1}\mu_0\Big) \\[0.3em]
&= \log\frac{p}{1-p}
\;-\;\frac12(\mu_1+\mu_0)^\top\Sigma^{-1}(\mu_1-\mu_0).
\end{aligned}




In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import multivariate_normal

def sigmoid(z): 
    return 1/(1+np.exp(-z))


class BayesClassifier:
    def __init__(self, p, mu0, mu1, rho):
        self.p = float(p)
        self.mu0 = np.asarray(mu0)
        self.mu1 = np.asarray(mu1)
        Sigma = np.array([[1, rho], [rho, 1]])
        S_inv = np.linalg.inv(Sigma)
        
        # direction vector: w = Sigma^{-1} (mu1 - mu0)
        self.w = S_inv @ (self.mu1 - self.mu0)
        
        # bias term
        t1 = self.mu1 @ S_inv @ self.mu1
        t0 = self.mu0 @ S_inv @ self.mu0
        self.b = np.log(self.p / (1 - self.p)) - 0.5 * (t1 - t0)

    def decision_function(self, X):
        """Return the linear score w^T x + b."""
        return np.asarray(X) @ self.w + self.b

    def predict_proba(self, X):
        s = self.decision_function(X)
        p1 = sigmoid(s)
        return np.c_[1 - p1, p1]

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


In [ ]:
def plot_mixture_and_bayes(p, mu0, mu1, rho):
    mu0, mu1 = map(np.asarray, (mu0, mu1))
    Sigma = np.array([[1, rho], [rho, 1]])
    clf = BayesClassifier(p, mu0, mu1, rho)

    # Grid (auto-sized if samples provided)
    x_min, x_max, y_min, y_max = -4, 4, -4, 4

    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                         np.linspace(y_min, y_max, 300))
    G = np.c_[xx.ravel(), yy.ravel()]

    # Gaussian mixture
    f0 = multivariate_normal(mu0, Sigma).pdf(G)
    f1 = multivariate_normal(mu1, Sigma).pdf(G)
    Z = ((1-p)*f0 + p*f1).reshape(xx.shape)

    # Bayes boundary
    post = clf.predict_proba(G)[:,1].reshape(xx.shape)

    # Plot
    plt.figure(figsize=(7,6))
    plt.contourf(xx, yy, Z, levels=20, cmap="viridis")
    plt.contour(xx, yy, Z, colors="k", linewidths=0.4)
    plt.contour(xx, yy, post, levels=[0.5], colors=["#ff0000"], linewidths=2.5)
    
    plt.gca().set_aspect("equal")
    plt.title(f"Mixture level sets + Bayes boundary (p={p}, rho={rho})")
    plt.show()


### Example: Visualizing the Bayes classifier

We now visualize two scenarios:
- balanced and uncorrelated mixture,
- unbalanced and correlated mixture.

Observe how the Bayes boundary changes.


In [ ]:
plot_mixture_and_bayes(
    p=0.5, 
    mu0=[-1.25, 0], 
    mu1=[+1.25, 0], 
    rho=0.0
)

# Now your turn, with unbalanced and correlated features
plot_mixture_and_bayes(
    p=0.25,
    mu0=[-1.25, 0],
    mu1=[+1.25, 0],
    rho=0.6
)


### Examples
Case (1) : balanced and uncorrelated mixture, $p=0.5,\Sigma_1 = \Sigma_2 = I_d$ <br> (orthogonal to $\mu_1 − \mu_0$, cross the middle point )<br> 
Case (2): unbalanced and uncorrelated mixture $p \neq 0.5,\Sigma_1 = \Sigma_2 = I_d$ <br> (orthogonal to $\mu_1 − \mu_0$, a shift)  <br> 
Case (3):  balanced and correlated mixture $p = 0.5,\Sigma_1 = \Sigma_2 \neq I_d$ <br> (cross the middle point of $\mu_1 − \mu_0$, not orthogonal , with a rotation  )  <br> 


In [ ]:
#  Case 1 : balanced and uncorrelated  (p=0.5, Sigma=I)
plot_mixture_and_bayes(
    p=0.5, 
    mu0=[-1.25, 0], 
    mu1=[+1.25, 0], 
    rho=0.0
)

#  Case 2 : unbalanced and uncorrelated  (p != 0.5, Sigma=I)
#  boundary stays orthogonal to mu1-mu0 but is shifted toward the rarer class
plot_mixture_and_bayes(
    p=0.2,
    mu0=[-1.25, 0],
    mu1=[+1.25, 0],
    rho=0.0
)

#  Case 3 : balanced and correlated  (p=0.5, Sigma != I)
#  boundary crosses the midpoint but is no longer orthogonal to mu1-mu0 (rotated)
plot_mixture_and_bayes(
    p=0.5,
    mu0=[-1.25, 0],
    mu1=[+1.25, 0],
    rho=0.7
)


## 2. Generating Data From a Gaussian Mixture

### Task
- Understand the `generate_mixture_samples` function.
- Generate datasets for multiple parameter choices.


In [ ]:
def generate_mixture_samples(n, p, mu0, mu1, rho, random_state=None):
    """
    Generate n samples from the 2D Gaussian mixture:
        (1-p) N(mu0, Sigma) + p N(mu1, Sigma)
    with Sigma = [[1, rho], [rho, 1]].
    Returns X (n,2) and y (n,) in {0,1}.
    """
    rng = np.random.default_rng(random_state)
    mu0, mu1 = np.asarray(mu0), np.asarray(mu1)
    Sigma = np.array([[1, rho], [rho, 1]])
    
    # sample class labels
    y = rng.binomial(1, p, size=n)
    
    # sample from multivariate normal according to classes
    X = np.zeros((n, 2))
    idx1 = (y == 1)
    idx0 = ~idx1
    
    if idx0.any():
        X[idx0] = multivariate_normal(mu0, Sigma).rvs(size=idx0.sum(), random_state=rng)
    if idx1.any():
        X[idx1] = multivariate_normal(mu1, Sigma).rvs(size=idx1.sum(), random_state=rng)
    
    return X, y


### Visualizing the sampled dataset

We now generate 500 samples and visualize the resulting clusters.
This will serve as our training data for LDA and logistic regression later on.


In [ ]:
p   = 0.5
mu0 = [-1.25, 0]
mu1 = [ 1.25, 0]
rho = 0.0

X, y = generate_mixture_samples(500, p, mu0, mu1, rho, random_state=0)

plt.figure(figsize=(6, 5))
plt.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", s=12, alpha=0.7)
plt.gca().set_aspect("equal")
plt.title(f"Gaussian mixture samples (n=500, p={p}, rho={rho})")
plt.xlabel("$x_1$")
plt.ylabel("$x_2$")
plt.show()


## 3. Linear Discriminant Analysis (LDA)

LDA is a **generative binary classifier** based on the assumption:

- both classes are Gaussian,
- with the **same covariance matrix** but different means.

Under these assumptions, the Bayes classifier is linear and can be estimated from data.

### Task
- Complete the LDA classifier below.
- Understand how `mu0`, `mu1`, and the shared covariance are estimated.

\begin{align}
\hat p 
&= \frac{1}{n}\sum_{i=1}^n \mathbf 1\{y_i = 1\}, \\
\hat\mu_k 
&= \frac{1}{n_k}\sum_{i:\, y_i = k} x_i,
\qquad k\in\{0,1\}, \\
\hat\Sigma 
&= \frac{1}{n}\sum_{k=0}^1 \sum_{i:\, y_i = k}
(x_i-\hat\mu_k)(x_i-\hat\mu_k)^\top,
\end{align}



In [ ]:
class LDAClassifier:
    def fit(self, X, y):
        X, y = np.asarray(X, float), np.asarray(y, int)
        
        X0, X1 = X[y == 0], X[y == 1]
        n0, n1 = len(X0), len(X1)
        self.p_ = n1 / (n0 + n1)
        
        # means
        self.mu0_, self.mu1_ = X0.mean(axis=0), X1.mean(axis=0)
        
        # pooled covariance (MLE, normalized by total n = n0 + n1)
        D0 = X0 - self.mu0_
        D1 = X1 - self.mu1_
        S = (D0.T @ D0 + D1.T @ D1) / (n0 + n1)
        S_inv = np.linalg.inv(S)
        
        # linear discriminant parameters
        self.w_ = S_inv @ (self.mu1_ - self.mu0_)
        t1 = self.mu1_ @ S_inv @ self.mu1_
        t0 = self.mu0_ @ S_inv @ self.mu0_
        self.b_ = np.log(self.p_ / (1 - self.p_)) - 0.5 * (t1 - t0)

        return self

    def decision_function(self, X):
        """Return the linear score w^T x + b."""
        return np.asarray(X) @ self.w_ + self.b_

    def predict_proba(self, X):
        p1 = sigmoid(self.decision_function(X))
        return np.c_[1 - p1, p1]

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


### Training and Evaluating LDA

We split the dataset into **training** and **test** sets.

### Task
- Train LDA using the training set.
- Evaluate prediction accuracy on the test set. 
- Print predicted probabilities for a few examples.
- Understand the structure of the predicted probabilities


In [ ]:
# --- generate data from mixture ---
p   = 0.5
mu0 = [-1.25, 0]
mu1 = [ 1.25, 0]
rho = 0.3

X, y = generate_mixture_samples(500, p, mu0, mu1, rho, random_state=0)

# --- train/test split ---
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0
)

# --- fit LDA ---
lda = LDAClassifier().fit(X_train, y_train)

# --- predict on first 5 test points ---
print("Test points:\n", X_test[:5])
print("True labels :", y_test[:5])
print("Predicted   :", lda.predict(X_test[:5]))
print("Predict proba:\n", lda.predict_proba(X_test[:5]))


### Comparing LDA and Bayes Boundaries

Here we compare the decision boundaries:
- **LDA** (estimated from finite data)
- **Bayes** (the true optimal classifier)

### Question
- When do you expect LDA to match the Bayes classifier?
- What happens when the sample size is small?


In [ ]:
bayes = BayesClassifier(p, mu0, mu1, rho)

# --- grid (based on train set) ---
x_min, x_max = X_train[:,0].min()-1, X_train[:,0].max()+1
y_min, y_max = X_train[:,1].min()-1, X_train[:,1].max()+1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                     np.linspace(y_min, y_max, 300))
G = np.c_[xx.ravel(), yy.ravel()]

post_lda   = lda.predict_proba(G)[:,1].reshape(xx.shape)
post_bayes = bayes.predict_proba(G)[:,1].reshape(xx.shape)

# --- plot ---
plt.figure(figsize=(7,6))
plt.scatter(X_train[:,0], X_train[:,1], c=y_train, cmap="coolwarm",s=15, alpha=0.75)

# LDA boundary (blue)
plt.contour(xx, yy, post_lda, levels=[0.5], colors=["#0000ff"], linewidths=2, linestyles="--")
# Bayes boundary (bright red)
plt.contour(xx, yy, post_bayes, levels=[0.5], colors=["#ff0000"], linewidths=2)

plt.gca().set_aspect("equal")
plt.title("Train samples + LDA (blue dashed) vs Bayes (red) boundary")
plt.show()


### Answer — When does LDA match the Bayes classifier?

LDA estimates exactly the same *form* of model as the Bayes rule in this setting — two Gaussians sharing one covariance — so its only "error" is statistical: it plugs in the empirical estimates $\hat p, \hat\mu_0, \hat\mu_1, \hat\Sigma$ in place of the true parameters. LDA converges to the Bayes classifier when:

- the modeling assumptions actually hold (both classes Gaussian with a **common** covariance), **and**
- the sample size is large, so $\hat\mu_k \to \mu_k$, $\hat\Sigma \to \Sigma$, $\hat p \to p$ by the law of large numbers.

In the limit $n\to\infty$ the estimated boundary coincides with the true Bayes boundary, and LDA is asymptotically optimal (Bayes-consistent) here.

**When sample size is small:** With few points the parameter estimates are noisy — especially $\hat\Sigma$ and the class means. The estimated boundary wobbles around the true one (higher variance), can be visibly rotated/shifted from the red Bayes line, and the gap is worse when one class is rare (small $n_k$ makes $\hat\mu_k$ and $\hat p$ unreliable) or when $\hat\Sigma$ is close to singular. Bias is small (correct model), but variance is large, so test accuracy sits below the Bayes-optimal rate until more data is collected.


Evaluate the accuracy of the LDA classifier on the test set

In [ ]:
# LDA accuracy on test set
acc_lda = (lda.predict(X_test) == y_test).mean()
print(f"LDA test accuracy: {acc_lda:.3f}")


### Double moons

We now use the double moons dataset, whose classes are clearly non-Gaussian.
Although LDA’s assumptions do not hold here, we can still train it and observe the resulting decision boundary and compare it with logistic regression.


In [ ]:
from sklearn.datasets import make_moons
import matplotlib.pyplot as plt

# generate data
X, y = make_moons(n_samples=1000, noise=0.2, random_state=0)

# scatter plot
plt.figure(figsize=(6,5))
plt.scatter(X[:,0], X[:,1], c=y, cmap="coolwarm", s=10)
plt.gca().set_aspect("equal")
plt.title("Two Moons Dataset (1000 samples)")
plt.show()


In [ ]:
from sklearn.model_selection import train_test_split

# train / test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0
)

# fit LDA
lda = LDAClassifier().fit(X_train, y_train)

# accuracy on test set
acc_lda = (lda.predict(X_test) == y_test).mean()
print(f"LDA test accuracy (two moons): {acc_lda:.3f}")


Now let us visualize the decision boundary

In [ ]:
def plot_points_and_boundary(clf, X, y, title="Decision boundary"):
    X, y = np.asarray(X), np.asarray(y)

    # grid around data
    x_min, x_max = X[:,0].min()-0.5, X[:,0].max()+0.5
    y_min, y_max = X[:,1].min()-0.5, X[:,1].max()+0.5
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 300),
        np.linspace(y_min, y_max, 300)
    )
    G = np.c_[xx.ravel(), yy.ravel()]

    # posterior for class 1 (assuming sklearn-style predict_proba)
    post = clf.predict_proba(G)[:,1].reshape(xx.shape)

    # plot
    plt.figure(figsize=(6,5))
    plt.scatter(X[:,0], X[:,1], c=y, cmap="coolwarm",
                s=12, alpha=0.7)
    plt.contour(xx, yy, post, levels=[0.5], colors=["#0000ff"], linewidths=2)

    plt.gca().set_aspect("equal")
    plt.title(title)
    plt.show()


In [ ]:
plot_points_and_boundary(lda, X_train, y_train, title="LDA decision boundary")


## 4. Logistic Regression

We now switch to a **discriminative** model: logistic regression.

### Objectives
- Fit a logistic classifier to the mixture data.
- Compare its decision boundary with LDA and Bayes.
- Explore improvements using **polynomial feature expansion**.


In [ ]:
from sklearn.linear_model import LogisticRegression

# --- fit Logistic Regression ---
lr = LogisticRegression().fit(X_train, y_train)

# --- accuracy ---
acc_lr = lr.score(X_test, y_test)
print(f"Logistic Regression test accuracy: {acc_lr:.3f}")

# --- display decision boundary ---
plot_points_and_boundary(lr, X_train, y_train,
                         title="Logistic Regression decision boundary (two moons)")


In [ ]:
from sklearn.preprocessing import PolynomialFeatures

k = 3  # polynomial degree

# --- polynomial transform ---
poly = PolynomialFeatures(k, include_bias=False)
X_train_poly = poly.fit_transform(X_train)
X_test_poly  = poly.transform(X_test)

# --- fit LR on polynomial features ---
lr_poly = LogisticRegression(max_iter=5000).fit(X_train_poly, y_train)

# --- accuracy ---
acc_lr_poly = lr_poly.score(X_test_poly, y_test)
print(f"Polynomial LR (degree {k}) test accuracy: {acc_lr_poly:.3f}")

# --- decision boundary plot (on original X grid) ---
# we wrap clf so plot_points_and_boundary can call predict_proba on raw X

class PolyWrapper:
    def __init__(self, poly, clf):
        self.poly = poly
        self.clf = clf
    def predict_proba(self, X):
        return self.clf.predict_proba(self.poly.transform(X))

plot_points_and_boundary(
    PolyWrapper(poly, lr_poly),
    X_train,
    y_train,
    title=f"Polynomial LR (degree {k}) decision boundary"
)


### Task: Increase the polynomial degree

Increase the value of \(k\) in the polynomial features and observe how the decision boundary evolves.  
To clearly see the effect **without regularization**, do **not** use the default scikit-learn setting.  
Instead, set: C = 1e9, so that regularization is essentially removed.  
Discuss what happens as \(k\) becomes large.

In [ ]:
# Effect of increasing the polynomial degree WITHOUT regularization (C = 1e9).
# We track train vs test accuracy and plot the boundary for each degree.

for k in [1, 3, 6, 10, 15]:
    poly_k = PolynomialFeatures(k, include_bias=False)
    Xtr_k = poly_k.fit_transform(X_train)
    Xte_k = poly_k.transform(X_test)

    # C = 1e9  ->  regularization essentially removed
    clf_k = LogisticRegression(C=1e9, max_iter=20000).fit(Xtr_k, y_train)

    train_acc = clf_k.score(Xtr_k, y_train)
    test_acc  = clf_k.score(Xte_k, y_test)
    print(f"degree {k:2d}:  train acc = {train_acc:.3f}   test acc = {test_acc:.3f}")

    plot_points_and_boundary(
        PolyWrapper(poly_k, clf_k),
        X_train, y_train,
        title=f"Poly LR degree {k} (C=1e9, no regularization)"
    )


### What happens as the degree $k$ grows

- **Low degree (k = 1):** the model is regular logistic regression — straight line. It underfits the two-moons shape (high bias) and leaves many points misclassified.
- **Moderate degree (k ≈ 3–6):** the boundary curves to follow the two crescents, training and test accuracy are both high and close to each other.
- **High degree (k ≳ 10), no regularization (C = 1e9):**  feature space becomes very high-dimensional, so the model has enough capacity to bend the boundary around individual noisy points. Training accuracy comes toward 1.0 while test accuracy eventually drops; the gap between them is sign of overfitting

So increasing $k$ trades bias for variance. Without the regularization that scikit-learn applies by default (disabled it with `C=1e9`), the high-degree model has nothing stopping it from fitting noise, which is why regularization (a smaller `C`) is normally kept on.


## 5. ROC Curves and AUC

Accuracy alone is often insufficient.  
We now compare models using **ROC curves** and the **Area Under the Curve (AUC)**.

### Models compared:
- LDA
- Logistic regression
- Logistic regression with polynomial features (degree 3)

### Task
- Generate ROC curves for the three models.
- Compare their AUCs.
- Comment on which model is best and why.


In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

# --- (re)fit the three models on the current train set ---
lda = LDAClassifier().fit(X_train, y_train)
lr  = LogisticRegression(max_iter=5000).fit(X_train, y_train)

poly3  = PolynomialFeatures(3, include_bias=False)
Xtr3   = poly3.fit_transform(X_train)
Xte3   = poly3.transform(X_test)
lr_poly3 = LogisticRegression(max_iter=5000).fit(Xtr3, y_train)

# --- class-1 scores on the test set ---
scores = {
    "LDA":             lda.predict_proba(X_test)[:, 1],
    "Logistic":        lr.predict_proba(X_test)[:, 1],
    "Poly LR (deg 3)": lr_poly3.predict_proba(Xte3)[:, 1],
}

# --- ROC curves ---
plt.figure(figsize=(7, 6))
for name, s in scores.items():
    fpr, tpr, _ = roc_curve(y_test, s)
    auc = roc_auc_score(y_test, s)
    plt.plot(fpr, tpr, lw=2, label=f"{name}  (AUC = {auc:.3f})")

plt.plot([0, 1], [0, 1], "k--", alpha=0.5, label="random")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC curves on the test set")
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.gca().set_aspect("equal")
plt.show()


### Comment — which model is best and why

On the **two-moons** data the ranking by AUC is:

$$\text{Poly LR (deg 3)} \;>\; \text{Logistic} \;\approx\; \text{LDA}.$$

- **LDA** and **plain logistic regression** are both *linear* classifiers, so their decision boundary is a single straight line. The two moons are not linearly separable, so both leave a chunk of each crescent on the wrong side — their ROC curves sit lower and their AUCs are the smallest (and close to each other).
- **Polynomial logistic regression (degree 3)** can bend the boundary to wrap around the crescents, so it separates the classes much better. Its ROC curve dominates (lies above/left of) the others and it has the **highest AUC**.

AUC is preferable to raw accuracy here because it is **threshold-independent** — it measures how well each model *ranks* positives above negatives across all decision thresholds, and it is insensitive to class imbalance. The degree-3 polynomial model wins because its hypothesis class matches the curved geometry of the data, while staying low enough in degree to avoid the overfitting we saw at large $k$.


## 6. Implementing Logistic Regression From Scratch

To deepen your understanding, you will build your own logistic regression class.

### Task
- Implement the gradient descent.


In [ ]:
import numpy as np

class myLogisticRegression:
    def __init__(self, lr=0.1, max_iter=1000):
        self.lr = lr
        self.max_iter = max_iter
        self.loss_history = []       # <--- NEW

    def sigmoid(self, z):
        return 1 / (1 + np.exp(-z))
    
    def fit(self, X, y):
        X = np.asarray(X, float)
        y = np.asarray(y, float)
        n, d = X.shape

        # add intercept
        Xb = np.c_[np.ones(n), X]
        self.w = np.zeros(d + 1)

        eps = 1e-12  # numerical guard for the log
        for _ in range(self.max_iter):
            scores = Xb @ self.w
            p1 = self.sigmoid(scores)

            # Binary cross-entropy loss (averaged over samples)
            loss = -np.mean(y * np.log(p1 + eps) + (1 - y) * np.log(1 - p1 + eps))
            self.loss_history.append(loss)

            # Gradient of the mean cross-entropy:  (1/n) X^T (p - y)
            grad = Xb.T @ (p1 - y) / n
            self.w -= self.lr * grad

        return self
    
    def predict_proba(self, X):
        Xb = np.c_[np.ones(len(X)), X]
        p1 = self.sigmoid(Xb @ self.w)
        return np.c_[1 - p1, p1]
    
    def predict(self, X):
        return (self.predict_proba(X)[:,1] >= 0.5).astype(int)


### Training Loss Curve

We now train our custom logistic model and visualize the **training loss**.

### Question
- Is the loss decreasing monotonically?
- What happens if you use a learning rate that’s too large? too small?


In [ ]:
clf = myLogisticRegression(lr=0.1, max_iter=3000)
clf.fit(X_train, y_train)

import matplotlib.pyplot as plt
plt.plot(clf.loss_history)
plt.xlabel("Iteration")
plt.ylabel("Train loss (cross-entropy)")
plt.title("Training loss curve")
plt.grid(True)
plt.show()


### Answer — reading training-loss curve

**Is the loss decreasing monotonically?** Yes. The binary cross-entropy loss for logistic regression is **convex** in $w$, and with a small enough, fixed learning rate full-batch gradient descent decreases the loss at every step. The curve falls steeply at first (large gradients) and then flattens as it approaches the minimum — a smooth, monotone decrease with no oscillations here.

**Learning rate too large.** Each update overshoots the minimum. The loss bounces up and down instead of decreasing monotonically, and if it is far too large the iterates **diverge** (loss blows up, `exp` overflows in the sigmoid). You would lower `lr`.

**Learning rate too small.** Each step barely moves the weights, so convergence is extremely slow: the loss curve creeps down and may not reach the minimum within `max_iter`. The fit is still correct in the limit, just inefficient — you would raise `lr` or increase `max_iter`.

I'd say best course of action is the largest/highest learning rate that still gives a smooth, monotone decrease.


Evaluate the accuracy

In [ ]:
# Accuracy of our from-scratch logistic regression on the test set
acc_mine = (clf.predict(X_test) == y_test).mean()
print(f"myLogisticRegression test accuracy: {acc_mine:.3f}")

# sanity check vs scikit-learn's logistic regression
acc_sklearn = LogisticRegression().fit(X_train, y_train).score(X_test, y_test)
print(f"sklearn LogisticRegression test accuracy: {acc_sklearn:.3f}")
